# Mitra — Telco Churn

Mitra é um modelo tabular fundacional da Amazon (2025), baseado em Transformer com ~72M parâmetros,
pré-treinado em dados sintéticos via in-context learning. Disponível no AutoGluon 1.4+.

São comparados dois modos:
- **Zero-shot**: sem fine-tuning, inferência direta via ICL.
- **Fine-tuned**: ajuste fino dos pesos no dataset de treino.

O melhor modo (KS no val) é usado para avaliação final no teste.

> **Instalação:** `uv add "autogluon.tabular[mitra]"`

In [1]:
import sys, os
sys.path.insert(0, '.')
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import joblib
from sklearn import metrics as skm
from importlib.metadata import version as pkg_version
from autogluon.tabular import TabularPredictor
from model_utils import load_data, compute_metrics, save_results, print_scores

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
AG_VERSION = pkg_version('autogluon.tabular')
print(f'AutoGluon version: {AG_VERSION}')

AutoGluon version: 1.5.0


## 1. Carregar dados

In [2]:
X_train, y_train, X_val, y_val, X_test, y_test = load_data()
print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')

yvl_np  = y_val.values.astype(int)
yte_np  = y_test.values.astype(int)

LABEL = 'Churn'

# AutoGluon exige DataFrame com coluna label incluída
train_df = X_train.copy(); train_df[LABEL] = y_train.values
val_df   = X_val.copy();   val_df[LABEL]   = y_val.values

print(f'Churn rate treino: {y_train.mean():.3f}')

Train: (7242, 22)  Val: (1057, 22)  Test: (1057, 22)
Churn rate treino: 0.500


## 2. Modo Zero-Shot (sem fine-tuning)

In [ ]:
os.makedirs('results/mitra/ag_zeroshot', exist_ok=True)

predictor_zs = TabularPredictor(
    label=LABEL,
    eval_metric='roc_auc',
    path='results/mitra/ag_zeroshot',
    verbosity=1,
)
predictor_zs.fit(
    train_df,
    hyperparameters={"MITRA": {"fine_tune": False, "ag.max_memory_usage_ratio": 100}},
    fit_weighted_ensemble=False,
)

probs_zs_val = predictor_zs.predict_proba(X_val)[1].values
fpr, tpr, _ = skm.roc_curve(yvl_np, probs_zs_val)
ks_zs = float(np.max(tpr - fpr))
print(f'Zero-shot  →  KS (val): {ks_zs:.4f}')

AutoGluon infers your prediction problem is: 'binary' (because only two unique label-values observed).
	If 'binary' is not the correct problem_type, please manually specify the problem_type parameter during Predictor init (You may specify problem_type as one of: ['binary', 'multiclass', 'regression', 'quantile'])


Zero-shot  →  KS (val): 0.5285


## 3. Modo Fine-Tuned

Busca simples sobre `fine_tune_steps` para maximizar KS no val.

In [ ]:
ft_steps_grid = [10, 25, 50, 100, 200]
ft_results = []

for steps in ft_steps_grid:
    path = f'results/mitra/ag_ft_{steps}'
    os.makedirs(path, exist_ok=True)
    pred = TabularPredictor(
        label=LABEL,
        eval_metric='roc_auc',
        path=path,
        verbosity=1,
    )
    pred.fit(
        train_df,
        hyperparameters={"MITRA": {"fine_tune": True, "fine_tune_steps": steps, "ag.max_memory_usage_ratio": 100}},
        fit_weighted_ensemble=False,
    )
    probs_val = pred.predict_proba(X_val)[1].values
    fpr, tpr, _ = skm.roc_curve(yvl_np, probs_val)
    ks = float(np.max(tpr - fpr))
    ft_results.append({'steps': steps, 'ks': ks, 'predictor': pred, 'probs_val': probs_val})
    print(f'fine_tune_steps={steps:>3d}  →  KS (val): {ks:.4f}')

best_ft = max(ft_results, key=lambda r: r['ks'])
print(f'\nMelhor fine-tuned  →  steps={best_ft["steps"]}  KS={best_ft["ks"]:.4f}')

AutoGluon infers your prediction problem is: 'binary' (because only two unique label-values observed).
	If 'binary' is not the correct problem_type, please manually specify the problem_type parameter during Predictor init (You may specify problem_type as one of: ['binary', 'multiclass', 'regression', 'quantile'])


## 4. Selecionar melhor modo e avaliar no teste

In [ ]:
if ks_zs >= best_ft['ks']:
    best_mode = 'zero-shot'
    best_predictor = predictor_zs
    print(f'Modo selecionado: zero-shot  (KS val={ks_zs:.4f})')
else:
    best_mode = f'fine-tuned (steps={best_ft["steps"]})'
    best_predictor = best_ft['predictor']
    print(f'Modo selecionado: fine-tuned steps={best_ft["steps"]}  (KS val={best_ft["ks"]:.4f})')

probs = best_predictor.predict_proba(X_test)[1].values
preds = (probs >= 0.5).astype(int)

scores = compute_metrics(yte_np, preds, probs)
print('\nScores no teste:')
print_scores(scores)

os.makedirs('results/mitra', exist_ok=True)
joblib.dump({'mode': best_mode, 'scores': scores}, 'results/mitra/model.pkl')

save_results('mitra', yte_np, preds, probs, scores)
print(f'\nNota: Mitra ({best_mode}), AutoGluon {AG_VERSION}.')